In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.utils import resample
from scipy.stats import wilcoxon
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [3]:
FEATURE_NAMES = (
    ['n_C','n_N','n_O','n_S','n_F','n_Cl','n_Br','n_I','n_P'] +
    ['n_double','n_triple','n_aromatic','n_rings','n_branches','n_charged'] +
    ['mol_weight','logP_proxy','HBD','HBA','ro5_passed'] +
    ['ro5_mw','ro5_logp','ro5_hbd','ro5_hba'] +
    ['smiles_len','unique_atoms','frac_aromatic'] +
    [f'bigram_{i}' for i in range(27)]   # <-- 27 not 28
)

In [4]:
# Load from the CSV you just exported
df_loaded = pd.read_csv("feature_matrix.csv")

X = df_loaded[FEATURE_NAMES].values      # (2035, 54)
y = df_loaded['BBB_label'].values        # (2035,)

print(f"X shape : {X.shape}")
print(f"y shape : {y.shape}")
print(f"BBB+    : {(y==1).sum()}  ({(y==1).mean()*100:.1f}%)")
print(f"BBB-    : {(y==0).sum()}  ({(y==0).mean()*100:.1f}%)")
print(f"Class prior pi = {(y==1).mean():.4f}")

X shape : (2035, 54)
y shape : (2035,)
BBB+    : 1559  (76.6%)
BBB-    : 476  (23.4%)
Class prior pi = 0.7661


In [5]:
# Split ONCE — this test set never changes across the entire simulation
X_train_pool, X_test_fixed, y_train_pool, y_test_fixed = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_SEED
)

print(f"\nTraining pool : {X_train_pool.shape[0]} molecules")
print(f"Fixed test set: {X_test_fixed.shape[0]} molecules")
print(f"Test BBB+     : {(y_test_fixed==1).sum()}  ({(y_test_fixed==1).mean()*100:.1f}%)")
print(f"Test BBB-     : {(y_test_fixed==0).sum()}  ({(y_test_fixed==0).mean()*100:.1f}%)")

# Class index partition from training pool only
idx_pos = np.where(y_train_pool == 1)[0]
idx_neg = np.where(y_train_pool == 0)[0]
pi      = len(idx_pos) / len(y_train_pool)

print(f"\nTraining pool BBB+ indices: {len(idx_pos)}")
print(f"Training pool BBB- indices: {len(idx_neg)}")
print(f"Class prior pi             : {pi:.4f}")


Training pool : 1628 molecules
Fixed test set: 407 molecules
Test BBB+     : 312  (76.7%)
Test BBB-     : 95  (23.3%)

Training pool BBB+ indices: 1247
Training pool BBB- indices: 381
Class prior pi             : 0.7660


In [6]:
def make_classifiers():
    """
    Returns fresh classifier instances per replication.
    LR: L2-regularised (Ridge), C selected to be stable at small n.
    DRF: Random Forest with OOB enabled for internal monitoring.
    """
    lr = LogisticRegression(
        penalty='l2',
        C=1.0,                    # tune if needed via cross-val
        solver='lbfgs',
        max_iter=1000,
        random_state=None         # deterministic given data
    )
    drf = RandomForestClassifier(
        n_estimators=200,
        max_features='sqrt',
        oob_score=True,
        n_jobs=-1,
        random_state=None         # varies per replication intentionally
    )
    return lr, drf

In [7]:
SAMPLE_SIZES = [49, 100, 200, 500, 1000, len(y_train_pool)]
B            = 1000

# Storage
results = {
    n: {'LR': [], 'DRF': [], 'delta': []}
    for n in SAMPLE_SIZES
}

print("Starting simulation...\n")
print(f"{'n':>6}  {'Rep':>5}  {'AUC_LR':>8}  {'AUC_DRF':>9}  {'Delta':>8}")
print("-" * 50)

for n in SAMPLE_SIZES:

    # Class-proportional subsample sizes from training pool
    n_pos = round(n * pi)
    n_neg = n - n_pos

    for b in range(B):

        # ── STEP 1: Stratified bootstrap from training pool ONLY ──
        boot_pos = resample(idx_pos, n_samples=n_pos,
                            replace=True, random_state=b)
        boot_neg = resample(idx_neg, n_samples=n_neg,
                            replace=True, random_state=b + 100000)
        boot_idx = np.concatenate([boot_pos, boot_neg])

        X_boot = X_train_pool[boot_idx]
        y_boot = y_train_pool[boot_idx]

        # ── STEP 2: Fit both classifiers on bootstrap sample ──────
        lr, drf = make_classifiers()
        lr.fit(X_boot, y_boot)
        drf.fit(X_boot, y_boot)

        # ── STEP 3: Evaluate on FIXED test set (never changes) ────
        auc_lr  = roc_auc_score(y_test_fixed,
                                lr.predict_proba(X_test_fixed)[:, 1])
        auc_drf = roc_auc_score(y_test_fixed,
                                drf.predict_proba(X_test_fixed)[:, 1])

        results[n]['LR'].append(auc_lr)
        results[n]['DRF'].append(auc_drf)
        results[n]['delta'].append(auc_drf - auc_lr)

    # Print progress every 100 reps (last rep shown)
    if (b + 1) % 100 == 0 or b == B - 1:
        print(f"  n={n:>5}  b={b+1:>4}  "
              f"LR={np.mean(results[n]['LR']):.4f}  "
              f"DRF={np.mean(results[n]['DRF']):.4f}  "
              f"Δ={np.mean(results[n]['delta']):+.4f}")

print("\nSimulation complete.")

Starting simulation...

     n    Rep    AUC_LR    AUC_DRF     Delta
--------------------------------------------------
  n=   49  b=1000  LR=0.8734  DRF=0.9030  Δ=+0.0296
  n=  100  b=1000  LR=0.8973  DRF=0.9228  Δ=+0.0256
  n=  200  b=1000  LR=0.9166  DRF=0.9373  Δ=+0.0206
  n=  500  b=1000  LR=0.9362  DRF=0.9516  Δ=+0.0154
  n= 1000  b=1000  LR=0.9440  DRF=0.9600  Δ=+0.0161
  n= 1628  b=1000  LR=0.9468  DRF=0.9646  Δ=+0.0178

Simulation complete.


In [8]:
summary_rows = []

for n in SAMPLE_SIZES:

    auc_lr  = np.array(results[n]['LR'])
    auc_drf = np.array(results[n]['DRF'])
    delta   = np.array(results[n]['delta'])

    # Wilcoxon signed-rank test on paired differences
    # zero_method='wilcox' drops exact zeros before ranking
    stat, p_val = wilcoxon(delta, zero_method='wilcox')

    re_n = np.var(auc_drf) / np.var(auc_lr)   # relative efficiency

    summary_rows.append({
        'n'              : n,
        'mean_AUC_LR'    : np.mean(auc_lr),
        'mean_AUC_DRF'   : np.mean(auc_drf),
        'var_LR'         : np.var(auc_lr),
        'var_DRF'        : np.var(auc_drf),
        'CI_LR_low'      : np.percentile(auc_lr, 2.5),
        'CI_LR_high'     : np.percentile(auc_lr, 97.5),
        'CI_DRF_low'     : np.percentile(auc_drf, 2.5),
        'CI_DRF_high'    : np.percentile(auc_drf, 97.5),
        'median_delta'   : np.median(delta),
        'RE_n'           : re_n,
        'wilcoxon_stat'  : stat,
        'p_value'        : p_val,
        'sig_DRF_wins'   : (p_val < 0.05) and (np.median(delta) > 0)
    })

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv("simulation_summary.csv", index=False)

# Print clean table
print(df_summary[[
    'n','mean_AUC_LR','mean_AUC_DRF',
    'var_LR','var_DRF','RE_n',
    'median_delta','p_value','sig_DRF_wins'
]].to_string(index=False, float_format=lambda x: f"{x:.5f}"))

   n  mean_AUC_LR  mean_AUC_DRF  var_LR  var_DRF    RE_n  median_delta  p_value  sig_DRF_wins
  49      0.87340       0.90301 0.00213  0.00052 0.24490       0.02296  0.00000          True
 100      0.89726       0.92283 0.00081  0.00020 0.24817       0.02136  0.00000          True
 200      0.91664       0.93728 0.00037  0.00009 0.24852       0.01830  0.00000          True
 500      0.93616       0.95155 0.00010  0.00004 0.38464       0.01452  0.00000          True
1000      0.94397       0.96003 0.00004  0.00002 0.49939       0.01600  0.00000          True
1628      0.94676       0.96458 0.00002  0.00001 0.48913       0.01776  0.00000          True


In [9]:
# n* = smallest n where DRF significantly beats LR
crossover = df_summary[df_summary['sig_DRF_wins'] == True]

if len(crossover) > 0:
    n_star = crossover['n'].iloc[0]
    print(f"\nCrossover point n* = {n_star}")
    print(f"At n={n_star}: mean AUC LR={crossover['mean_AUC_LR'].iloc[0]:.4f}, "
          f"DRF={crossover['mean_AUC_DRF'].iloc[0]:.4f}, "
          f"p={crossover['p_value'].iloc[0]:.4f}")
else:
    print("\nNo significant crossover found across tested sample sizes.")
    print("DRF does not significantly outperform LR at any tested n.")


Crossover point n* = 49
At n=49: mean AUC LR=0.8734, DRF=0.9030, p=0.0000
